# "Ale czekaj... to jeszcze nie koniec"

## Bardziej widoczny Agent Loop

Cyfrowy Bliźniak zawierał Agent Loop. Ale działał za kulisami, uruchamiając się za każdym razem, gdy użytkownik wysyłał wiadomość. Używał swoich narzędzi, a potem odpowiadał. Nie było w tym za bardzo... pętli.

### Dodajmy 2 kolejne składniki, żeby było bardziej realistycznie

Zróbmy Agent Loop z kilkoma znajomymi funkcjami zapożyczonymi z Claude Code:

1. Terminal UI (TUI)
2. Narzędzie Checklist do wywoływania i śledzenia wielu wywołań narzędzi


In [ ]:
# Zacznij od kilku importów - rich to biblioteka do tworzenia sformatowanego tekstu w terminalu

from rich.console import Console  # klasa do wypisywania sformatowanego tekstu w terminalu
from dotenv import load_dotenv  # wczytuje zmienne środowiskowe (klucze API) z pliku .env
from anthropic import Anthropic  # klient SDK Anthropic - odpowiednik `from openai import OpenAI`
import json  # do serializacji wyników narzędzi do formatu JSON
load_dotenv(override=True)  # ładuje .env i nadpisuje istniejące zmienne środowiskowe

In [ ]:
def show(text):  # pomocnicza funkcja do wypisywania tekstu w konsoli
    try:
        Console().print(text)  # próba wypisania z formatowaniem rich markup (np. [green], [strike])
    except Exception:
        print(text)  # fallback na zwykły print, gdy tekst nie jest poprawnym rich markup

In [ ]:
anthropic = Anthropic()  # inicjalizacja klienta Anthropic - klucz brany z ANTHROPIC_API_KEY w .env

In [ ]:
# Kilka list!

checklist = []  # lista treści zadań na checkliście
completed = []  # lista bool - status ukończenia dla każdego indeksu w checklist

In [ ]:
def get_checklist_report() -> str:  # buduje i wypisuje aktualny stan checklisty jako tekst
    result = ""  # akumulator na sformatowany raport
    for index, item in enumerate(checklist):  # iteruj po zadaniach z numeracją od 0
        if completed[index]:  # sprawdź status ukończenia danego zadania
            result += f"Checklista #{index + 1}: [green][strike]{item}[/strike][/green]\n"  # ukończone - przekreślone, na zielono
        else:
            result += f"Checklista #{index + 1}: {item}\n"  # nieukończone - zwykły tekst
    show(result)  # wypisz raport w terminalu
    return result  # zwróć tekst raportu (użyty też jako wynik narzędzia)

In [ ]:
get_checklist_report()  # wywołanie testowe - pokazuje pusty raport na starcie

In [ ]:
def create_checklist(descriptions: list[str]) -> str:  # narzędzie: dodaje nowe pozycje do checklisty
    checklist.extend(descriptions)  # dopisz opisy nowych zadań do wspólnej listy
    completed.extend([False] * len(descriptions))  # każde nowe zadanie startuje jako nieukończone
    return get_checklist_report()  # zwróć zaktualizowany raport (trafi do Claude jako tool_result)

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:  # narzędzie: oznacza zadanie jako ukończone
    if 1 <= index <= len(checklist):  # walidacja 1-indeksowanego numeru zadania
        completed[index - 1] = True  # oznacz zadanie jako ukończone (indeksy list liczone od 0)
    else:
        return "Nie ma checklisty pod tym indeksem."  # błędny indeks - zwracany do Claude jako wynik narzędzia
    Console().print(completion_notes)  # wypisz notatkę o sposobie ukończenia zadania
    return get_checklist_report()  # zwróć zaktualizowany raport

In [ ]:
checklist, completed = [], []  # reset stanu przed testem ręcznym

create_checklist(["Kup zakupy", "Skończ tydzień 1", "Zjedz banana"])  # przykładowe wywołanie testowe narzędzia

In [ ]:
mark_complete(1, "kupione")  # przykładowe wywołanie testowe - oznacza zadanie #1 jako ukończone

In [ ]:
create_checklist_json = {
    "name": "create_checklist",  # nazwa narzędzia - musi się zgadzać z nazwą funkcji Pythona wołanej w handle_tool_calls
    "description": "Dodaj nową checklistę z listy opisów i zwróć pełną listę",  # opis czytany przez Claude przy decyzji, kiedy użyć narzędzia
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Opisy pozycji checklisty'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",  # nazwa narzędzia - musi się zgadzać z nazwą funkcji Pythona
    "description": "Oznacz jako ukończoną pozycję checklisty pod podanym indeksem (liczonym od 1) i zwróć pełną listę",  # opis czytany przez Claude
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI
        'properties': {
            'index': {
                'description': 'Indeks pozycji checklisty do oznaczenia jako ukończona, liczony od 1',
                'title': 'Indeks',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notatka o sposobie ukończenia zadania, w formacie rich console markup',
                'title': 'Notatka o ukończeniu',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [create_checklist_json, mark_complete_json]  # Anthropic przyjmuje płaską listę definicji narzędzi, bez opakowania {"type": "function", "function": ...} jak w OpenAI

In [ ]:
def handle_tool_calls(tool_calls):  # wykonuje listę bloków tool_use i zwraca wyniki dla Claude
    results = []  # lista bloków tool_result do wysłania z powrotem
    for tool_call in tool_calls:  # iteruj po każdym żądaniu wywołania narzędzia
        tool_name = tool_call.name  # blok tool_use ma name bezpośrednio, nie zagnieżdżone w .function jak w OpenAI
        arguments = tool_call.input  # input jest już sparsowanym dict, nie JSON-stringiem jak tool_call.function.arguments w OpenAI
        tool = globals().get(tool_name)  # znajdź funkcję Pythona o tej samej nazwie co narzędzie
        result = tool(**arguments) if tool else {}  # wywołaj narzędzie z argumentami albo zwróć pusty wynik, gdy nie znaleziono
        results.append({
            "type": "tool_result",  # Anthropic: blok tool_result zamiast wiadomości z rolą "tool" jak w OpenAI
            "tool_use_id": tool_call.id,  # musi się zgadzać z id bloku tool_use, na który odpowiadamy
            "content": json.dumps(result)  # treść wyniku jako string JSON
        })
    return results  # zwracane bloki trafią razem do JEDNEJ wiadomości user

In [ ]:
def loop(messages):  # pętla agentowa: woła Claude, wykonuje narzędzia, powtarza aż dostaniemy tekst
    response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages, tools=tools)  # system jako osobny top-level parametr, nie wpis w messages jak w OpenAI
    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia (odpowiednik finish_reason == "tool_calls" w OpenAI)
        tool_calls = [block for block in response.content if block.type == "tool_use"]  # wyciągnij bloki tool_use z odpowiedzi
        results = handle_tool_calls(tool_calls)  # wykonaj narzędzia i zbierz wyniki
        messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant (razem z blokami tool_use) wraca do historii
        messages.append({"role": "user", "content": results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user - Claude oczekuje ich razem, nie po jednej na wiadomość
        response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages, tools=tools)  # kolejne zapytanie z zaktualizowaną historią
    show(next(block.text for block in response.content if block.type == "text"))  # wypisz finalną odpowiedź tekstową (odporne na to, że content[0] bywa ThinkingBlock)

In [ ]:
system_message = """
Otrzymujesz problem do rozwiązania - użyj swoich narzędzi checklisty, żeby zaplanować listę kroków, a potem wykonaj każdy krok po kolei.
Teraz stwórz plan, ustaw checklistę, wykonaj kroki i odpowiedz rozwiązaniem.
Jeśli w pytaniu brakuje jakiejś wielkości, dodaj krok polegający na przyjęciu rozsądnego oszacowania.
Podaj rozwiązanie w formacie rich console markup, bez bloków kodu.
Nie zadawaj użytkownikowi pytań ani nie proś o doprecyzowanie; odpowiedz wyłącznie wynikiem po użyciu swoich narzędzi.
"""  # prompt systemowy - trafi do parametru system=, nie do listy messages
user_message = """"
Pociąg wyjeżdża z Bostonu o 14:00, jadąc z prędkością 60 mph.
Inny pociąg wyjeżdża z Nowego Jorku o 15:00, jadąc z prędkością 80 mph w stronę Bostonu.
Kiedy się spotkają?
"""  # treść pytania do rozwiązania przez Agent Loop
messages = [{"role": "user", "content": user_message}]  # bez wpisu roli "system" - Anthropic przyjmuje system jako osobny parametr top-level

In [ ]:
checklist, completed = [], []  # zresetuj stan checklisty przed uruchomieniem Agent Loop
loop(messages)  # uruchom pętlę agentową (system_message użyty wewnątrz loop jako global)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">Spróbuj teraz sam zbudować Agent Loop od zera!<br/>
            Stwórz nowy .ipynb i zrób go od podstaw, wracając do tego notatnika w razie potrzeby.<br/>
            To jeden z niewielu przypadków, gdzie rekomenduję pisanie od zera - daje bardzo satysfakcjonujący efekt.
            </span>
        </td>
    </tr>
</table>